# CatVTON 가상 피팅 프로토타입

사람 사진 + 옷 사진 → 합성 이미지.

Colab / Kaggle 양쪽에서 동작합니다.
- **Colab**: 마지막 Gradio 셀까지 실행하면 공개 링크로 웹 UI 테스트 가능
- **Kaggle**: 노트북 설정에서 **Internet: On**, **Accelerator: GPU** 필수. Gradio 공개 링크는 막히므로 배치 추론 셀 사용

GPU는 16GB(T4/P100) 이상 권장 (1024×768 기준 약 8GB 사용).

## 1. 환경 확인

In [ ]:
!nvidia-smi
import sys, torch
print('python:', sys.version)
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

## 2. 저장소 clone + 의존성 설치

저장소의 `requirements.txt`는 torch 2.1.2로 고정돼 있지만, Colab/Kaggle에 이미 설치된 torch를 그대로 쓰는 게 안전합니다 (버전을 내리면 CUDA 불일치로 깨지는 경우가 많음). 그래서 torch·torchvision·xformers는 제외하고 설치합니다.

**설치 후 런타임 재시작이 필요할 수 있습니다** (numpy 버전이 바뀌므로). 재시작하라는 메시지가 뜨면 재시작 후 이 셀 아래부터 다시 실행하세요.

In [ ]:
import os

WORK_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
REPO_DIR = os.path.join(WORK_DIR, 'CatVTON')
os.chdir(WORK_DIR)

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Zheng-Chong/CatVTON.git
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

In [ ]:
!pip install -q \
    accelerate==0.31.0 \
    diffusers==0.29.2 \
    huggingface_hub==0.23.4 \
    transformers==4.27.3 \
    numpy==1.26.4 \
    opencv-python==4.10.0.84 \
    scikit-image==0.24.0 \
    scipy==1.13.1 \
    av \
    gradio==4.39.0

> 위 설치에서 `transformers==4.27.3` 때문에 의존성 충돌이 나면, `transformers==4.36.2` 정도로 올려서 다시 시도해 보세요. CatVTON은 CLIP 텍스트 인코더를 쓰지 않아 transformers 버전에 크게 민감하지 않습니다.

## 3. 모델 가중치 다운로드

HuggingFace에서 자동으로 받습니다 (CatVTON attention 가중치 + DensePose/SCHP 마스크 생성 모델). 처음 실행 시 수 GB 다운로드.

In [ ]:
from huggingface_hub import snapshot_download

BASE_MODEL = 'runwayml/stable-diffusion-inpainting'
repo_path = snapshot_download(repo_id='zhengchong/CatVTON')
print('CatVTON weights:', repo_path)

## 4. 파이프라인 초기화

In [ ]:
import torch
from diffusers.image_processor import VaeImageProcessor

from model.pipeline import CatVTONPipeline
from model.cloth_masker import AutoMasker, vis_mask
from utils import init_weight_dtype, resize_and_crop, resize_and_padding

WIDTH, HEIGHT = 768, 1024
MIXED_PRECISION = 'bf16'  # T4처럼 bf16 미지원 GPU면 'fp16'으로

pipeline = CatVTONPipeline(
    base_ckpt=BASE_MODEL,
    attn_ckpt=repo_path,
    attn_ckpt_version='mix',
    weight_dtype=init_weight_dtype(MIXED_PRECISION),
    use_tf32=True,
    device='cuda',
)

automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, 'DensePose'),
    schp_ckpt=os.path.join(repo_path, 'SCHP'),
    device='cuda',
)

mask_processor = VaeImageProcessor(
    vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True
)
print('ready')

## 5. 추론 함수

나중에 앱 서버를 붙일 때 이 함수를 그대로 재사용합니다.

**입력**: 인물 이미지, 옷 이미지, 옷 종류 / **출력**: 합성된 PIL 이미지

In [ ]:
from PIL import Image

CLOTH_TYPES = ['upper', 'lower', 'overall', 'inner', 'outer']


def try_on(person, garment, cloth_type='upper', steps=50, guidance_scale=2.5, seed=42):
    """person/garment: 파일 경로 또는 PIL.Image. cloth_type: CLOTH_TYPES 중 하나."""
    if isinstance(person, str):
        person = Image.open(person)
    if isinstance(garment, str):
        garment = Image.open(garment)
    person = resize_and_crop(person.convert('RGB'), (WIDTH, HEIGHT))
    garment = resize_and_padding(garment.convert('RGB'), (WIDTH, HEIGHT))

    mask = automasker(person, cloth_type)['mask']
    mask = mask_processor.blur(mask, blur_factor=9)

    generator = torch.Generator(device='cuda').manual_seed(seed) if seed != -1 else None
    result = pipeline(
        image=person,
        condition_image=garment,
        mask=mask,
        num_inference_steps=steps,
        guidance_scale=guidance_scale,
        generator=generator,
    )[0]
    return result, person, garment, mask

## 6. 배치 추론 (Colab / Kaggle 공통)

저장소에 포함된 데모 이미지로 먼저 동작을 확인합니다. 본인 사진으로 테스트할 때는 아래 경로만 바꾸세요.

In [ ]:
!ls resource/demo/example/person/men resource/demo/example/condition/upper

In [ ]:
import glob
from IPython.display import display

person_path = sorted(glob.glob('resource/demo/example/person/men/*'))[0]
garment_path = sorted(glob.glob('resource/demo/example/condition/upper/*'))[0]

result, person, garment, mask = try_on(person_path, garment_path, cloth_type='upper')

os.makedirs('outputs', exist_ok=True)
result.save('outputs/result_00.png')

for img in (person, garment, vis_mask(person, mask), result):
    display(img.resize((288, 384)))

### 여러 조합 한 번에 돌리기

본인 사진 여러 장 × 옷 여러 벌을 돌려 실패 케이스(측면 사진, 팔로 옷 가림, 특이 포즈)를 수집합니다.

In [ ]:
# 본인 사진을 업로드한 뒤 아래 리스트를 채우세요
persons = sorted(glob.glob('resource/demo/example/person/men/*'))[:2]
garments = sorted(glob.glob('resource/demo/example/condition/upper/*'))[:2]

for i, p in enumerate(persons):
    for j, g in enumerate(garments):
        result, *_ = try_on(p, g, cloth_type='upper')
        out = f'outputs/p{i}_g{j}.png'
        result.save(out)
        print(out, '|', os.path.basename(p), '+', os.path.basename(g))
        display(result.resize((288, 384)))

## 7. Gradio 웹 데모 (Colab 권장)

저장소에 내장된 데모를 그대로 띄웁니다. 출력에 나오는 `*.gradio.live` 링크로 접속해 사진을 업로드하세요.

> Kaggle에서는 공개 링크가 막히는 경우가 많으니, 위 6번 배치 추론을 쓰세요.

In [ ]:
!python app.py --output_dir="resource/demo/output" --mixed_precision="bf16" --allow_tf32